# HR Compensation Analysis — Python / Pandas
## 1. Data Loading


### Running this notebook
Open in Google Colab and upload `HR_Compensation_Raw_Data_200_rows.xlsm` using the first code cell. Required worksheets: `employees_raw`, `reference_tables`, and `cleaning_report_template`.
Dependencies: pandas, numpy, openpyxl, statsmodels.
Saved analysis outputs are retained; the upload widget must be run in your own Colab session. The source dataset is required to reproduce the analysis.


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

file_path = "HR_Compensation_Raw_Data_200_rows.xlsm"

excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

['employees_raw', 'reference_tables', 'cleaning_report_template', 'README']

In [ ]:
employees_raw = pd.read_excel(
    file_path,
    sheet_name="employees_raw",
    dtype=object
)

reference_tables = pd.read_excel(
    file_path,
    sheet_name="reference_tables",
    dtype=object
)

cleaning_report_template = pd.read_excel(
    file_path,
    sheet_name="cleaning_report_template",
    dtype=object
)

print(f"Кількість рядків: {employees_raw.shape[0]}")
print(f"Кількість колонок: {employees_raw.shape[1]}")

employees_raw.head()

Кількість рядків: 200
Кількість колонок: 11


,employee_id,full_name,department_id,department_name,job_title,salary,hire_date,gender,city,employment_status,performance_score
0,E1001,Iryna Lysenko,D06,Operations,Procurement Specialist,32000,2016-05-10,Female,lviv,active,6
1,E1002,Yurii Rudenko,D05,IT,Software Engineer,91000,15/10/2017,Unknown,REM,active,5
2,E1003,Oleksandr Kravchenko,D02,MKT,Marketing Specialist,"48,000",07/03/2024,чоловік,Kyiv,inactive,0
3,E1004,Anastasiia Koval,D01,SALES,Sales Representative,"29,000",04.08.2025,Female,Харків,Відпустка,4
4,E1005,Maksym Moroz,D05,Information Technology,Software Engineer,69000,27.12.2021,M,Одеса,Активний,2


In [ ]:
print("Dataset shape:", employees_raw.shape)
display(employees_raw.head())

Dataset shape: (200, 11)


,employee_id,full_name,department_id,department_name,job_title,salary,hire_date,gender,city,employment_status,performance_score
0,E1001,Iryna Lysenko,D06,Operations,Procurement Specialist,32000,2016-05-10,Female,lviv,active,6
1,E1002,Yurii Rudenko,D05,IT,Software Engineer,91000,15/10/2017,Unknown,REM,active,5
2,E1003,Oleksandr Kravchenko,D02,MKT,Marketing Specialist,"48,000",07/03/2024,чоловік,Kyiv,inactive,0
3,E1004,Anastasiia Koval,D01,SALES,Sales Representative,"29,000",04.08.2025,Female,Харків,Відпустка,4
4,E1005,Maksym Moroz,D05,Information Technology,Software Engineer,69000,27.12.2021,M,Одеса,Активний,2


## 2. Data Quality Assessment

Перевіряємо структуру сирих даних, пропуски, дублікати, типи даних і некоректні значення. Вихідний DataFrame `employees_raw` залишається незмінним.


### 2.1 Structure, Data Types, Missing Values and Duplicates

In [ ]:
print("РОЗМІР ТАБЛИЦІ")
print(f"Рядків: {employees_raw.shape[0]}")
print(f"Колонок: {employees_raw.shape[1]}")

print("\nНАЗВИ КОЛОНОК")
print(employees_raw.columns.tolist())

print("\nТИПИ ДАНИХ")
display(employees_raw.dtypes.to_frame(name="data_type"))

print("\nКІЛЬКІСТЬ ПРОПУСКІВ")
missing_values = employees_raw.isna().sum().to_frame(name="missing_count")
missing_values["missing_percent"] = (
    missing_values["missing_count"] / len(employees_raw) * 100
).round(1)

display(missing_values)

print("\nКІЛЬКІСТЬ ПОВНИХ ДУБЛІКАТІВ")
print(employees_raw.duplicated().sum())

РОЗМІР ТАБЛИЦІ
Рядків: 200
Колонок: 11

НАЗВИ КОЛОНОК
['employee_id', 'full_name', 'department_id', 'department_name', 'job_title', 'salary', 'hire_date', 'gender', 'city', 'employment_status', 'performance_score']

ТИПИ ДАНИХ


,data_type
employee_id,object
full_name,object
department_id,object
department_name,object
job_title,object
salary,object
hire_date,object
gender,object
city,object
employment_status,object



КІЛЬКІСТЬ ПРОПУСКІВ


,missing_count,missing_percent
employee_id,1,0.5
full_name,0,0.0
department_id,2,1.0
department_name,0,0.0
job_title,0,0.0
salary,2,1.0
hire_date,0,0.0
gender,39,19.5
city,0,0.0
employment_status,0,0.0



КІЛЬКІСТЬ ПОВНИХ ДУБЛІКАТІВ
1


### 2.2 Employee ID Validation

In [ ]:
# Додаємо номер рядка вихідного Excel для зручності перевірки
audit_data = employees_raw.copy()
audit_data["source_excel_row"] = audit_data.index + 2

# Повні дублікати
full_duplicates = audit_data[
    employees_raw.duplicated(keep=False)
]

print("ПОВНІ ДУБЛІКАТИ")
display(full_duplicates)

# Очищаємо ID тільки для перевірки, не змінюючи вихідну таблицю
employee_id_check = (
    employees_raw["employee_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# ID, які не відповідають формату E + 4 цифри
invalid_id_mask = ~employee_id_check.str.match(r"^E\d{4}$")

print("\nНЕКОРЕКТНІ АБО ПОРОЖНІ EMPLOYEE_ID")
display(
    audit_data.loc[
        invalid_id_mask,
        ["source_excel_row", "employee_id", "full_name"]
    ]
)

# Повторні непорожні ID
duplicate_id_mask = (
    employee_id_check.duplicated(keep=False)
    & employee_id_check.ne("")
)

print("\nПОВТОРНІ EMPLOYEE_ID")
display(
    audit_data.loc[
        duplicate_id_mask,
        [
            "source_excel_row",
            "employee_id",
            "full_name",
            "department_id",
            "job_title"
        ]
    ].sort_values("employee_id")
)

ПОВНІ ДУБЛІКАТИ


,employee_id,full_name,department_id,department_name,job_title,salary,hire_date,gender,city,employment_status,performance_score,source_excel_row
19,E1020,Viktoriia Lysenko,D06,Операції,Procurement Specialist,UAH 44000,03/28/2023,Ж,odesa,Active,5,21
199,E1020,Viktoriia Lysenko,D06,Операції,Procurement Specialist,UAH 44000,03/28/2023,Ж,odesa,Active,5,201



НЕКОРЕКТНІ АБО ПОРОЖНІ EMPLOYEE_ID


,source_excel_row,employee_id,full_name
101,103,E10A2,Taras Kozak
156,158,NaN,Taras Marchenko
187,189,,Oleksandr Savchenko



ПОВТОРНІ EMPLOYEE_ID


,source_excel_row,employee_id,full_name,department_id,job_title
9,11,E1010,Dmytro Rudenko,D08,Contract Manager
44,46,E1010,Mykhailo Kravchenko,D04,People Partner
45,47,E1010,Ivan Kozak,D07,Technical Support Specialist
19,21,E1020,Viktoriia Lysenko,D06,Procurement Specialist
199,201,E1020,Viktoriia Lysenko,D06,Procurement Specialist


### 2.3 Categorical Values Validation

In [ ]:
categorical_columns = [
    "department_name",
    "city",
    "gender",
    "employment_status"
]

for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"КОЛОНКА: {column}")
    print(f"Кількість унікальних значень: {employees_raw[column].nunique(dropna=True)}")

    display(
        employees_raw[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )


КОЛОНКА: department_name
Кількість унікальних значень: 40


,department_name,count
0,Information Technology,11
1,Юридичний,9
2,it,8
3,Операції,8
4,Tech,8
5,CS,8
6,Legl,7
7,Human Resources,7
8,Operational,7
9,People,6



КОЛОНКА: city
Кількість унікальних значень: 30


,city,count
0,Одеса,11
1,dnipro,11
2,Київ,11
3,Lwow,10
4,kyiv,9
5,Online,9
6,Kharkov,8
7,Віддалено,8
8,DNIPRO,8
9,lviv,8



КОЛОНКА: gender
Кількість унікальних значень: 14


,gender,count
0,NaN,39
1,чоловік,17
2,жінка,16
3,Ж,15
4,Male,15
5,M,14
6,Ч,12
7,Woman,12
8,female,11
9,Female,11



КОЛОНКА: employment_status
Кількість унікальних значень: 14


,employment_status,count
0,active,36
1,Active,35
2,ACTIVE,30
3,Активний,29
4,working,27
5,terminated,12
6,Terminated,10
7,inactive,5
8,Звільнений,4
9,maternity leave,4


### 2.4 Salary Validation and Outlier Detection

In [ ]:
def parse_salary(value):
    """Перетворює сире значення зарплати на число."""

    if pd.isna(value):
        return np.nan

    cleaned_value = str(value).strip()
    cleaned_value = cleaned_value.replace("\u00a0", "")
    cleaned_value = re.sub(
        r"UAH|грн|₴|\s|,",
        "",
        cleaned_value,
        flags=re.IGNORECASE
    )

    return pd.to_numeric(cleaned_value, errors="coerce")


salary_numeric = employees_raw["salary"].apply(parse_salary)

salary_audit = pd.DataFrame({
    "source_excel_row": employees_raw.index + 2,
    "employee_id": employees_raw["employee_id"],
    "job_title": employees_raw["job_title"],
    "salary_raw": employees_raw["salary"],
    "salary_numeric": salary_numeric
})

print("ПЕРЕВІРКА ЗАРПЛАТ")
print(f"Пропущені значення: {salary_numeric.isna().sum()}")
print(f"Нульові значення: {(salary_numeric == 0).sum()}")
print(f"Від'ємні значення: {(salary_numeric < 0).sum()}")

print("\nНУЛЬОВІ ТА ВІД’ЄМНІ ЗАРПЛАТИ")
display(
    salary_audit[
        salary_audit["salary_numeric"].notna()
        & (salary_audit["salary_numeric"] <= 0)
    ]
)

# Розрахунок меж викидів методом IQR тільки для додатних зарплат
positive_salary = salary_numeric[salary_numeric > 0]

q1 = positive_salary.quantile(0.25)
q3 = positive_salary.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

salary_outlier_mask = (
    (salary_numeric < lower_bound)
    | (salary_numeric > upper_bound)
)

print("\nМЕЖІ ВИКИДІВ ЗА МЕТОДОМ IQR")
print(f"Q1: {q1:,.0f}")
print(f"Q3: {q3:,.0f}")
print(f"IQR: {iqr:,.0f}")
print(f"Нижня межа: {lower_bound:,.0f}")
print(f"Верхня межа: {upper_bound:,.0f}")

print("\nПОТЕНЦІЙНІ ВИКИДИ")
display(salary_audit[salary_outlier_mask])

ПЕРЕВІРКА ЗАРПЛАТ
Пропущені значення: 2
Нульові значення: 2
Від'ємні значення: 2

НУЛЬОВІ ТА ВІД’ЄМНІ ЗАРПЛАТИ


,source_excel_row,employee_id,job_title,salary_raw,salary_numeric
11,13,E1012,Support Team Lead,0,0.0
42,44,E1043,PPC Specialist,-40506,-40506.0
65,67,E1066,Onboarding Specialist,0,0.0
140,142,E1141,Compliance Specialist,-42414,-42414.0



МЕЖІ ВИКИДІВ ЗА МЕТОДОМ IQR
Q1: 37,000
Q3: 57,000
IQR: 20,000
Нижня межа: 7,000
Верхня межа: 87,000

ПОТЕНЦІЙНІ ВИКИДИ


,source_excel_row,employee_id,job_title,salary_raw,salary_numeric
1,3,E1002,Software Engineer,91000,91000.0
11,13,E1012,Support Team Lead,0,0.0
42,44,E1043,PPC Specialist,-40506,-40506.0
65,67,E1066,Onboarding Specialist,0,0.0
94,96,E1095,Recruiter,250000,250000.0
129,131,E1130,Software Engineer,UAH 102000,102000.0
140,142,E1141,Compliance Specialist,-42414,-42414.0
172,174,E1173,Data Analyst,300 000 грн,300000.0
195,197,E1196,Finance Manager,88000,88000.0
197,199,E1198,Legal Counsel,90 000 грн,90000.0


### 2.5 Hire Date Validation

In [ ]:
analysis_date = pd.Timestamp("2026-09-01")


def parse_hire_date(value):
    """Розпізнає змішані формати дат за визначеними правилами."""

    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    try:
        # ISO: yyyy-mm-dd або yyyy/mm/dd
        if re.fullmatch(r"\d{4}[-/]\d{1,2}[-/]\d{1,2}", value):
            return pd.to_datetime(
                value.replace("/", "-"),
                format="%Y-%m-%d",
                errors="coerce"
            )

        # Європейський формат із крапками: dd.mm.yyyy
        if re.fullmatch(r"\d{1,2}\.\d{1,2}\.\d{4}", value):
            return pd.to_datetime(
                value,
                format="%d.%m.%Y",
                errors="coerce"
            )

        # Дати зі слешем
        if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", value):
            first, second, year = map(int, value.split("/"))

            if first > 12:
                date_format = "%d/%m/%Y"
            elif second > 12:
                date_format = "%m/%d/%Y"
            else:
                # Для неоднозначних дат приймаємо dd/mm/yyyy
                date_format = "%d/%m/%Y"

            return pd.to_datetime(
                value,
                format=date_format,
                errors="coerce"
            )

    except (ValueError, TypeError):
        return pd.NaT

    return pd.NaT


hire_date_parsed = employees_raw["hire_date"].apply(parse_hire_date)

# Виявлення неоднозначних дат зі слешем
def is_ambiguous_slash_date(value):
    if pd.isna(value):
        return False

    value = str(value).strip()

    if not re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", value):
        return False

    first, second, _ = map(int, value.split("/"))
    return first <= 12 and second <= 12


ambiguous_date_mask = employees_raw["hire_date"].apply(
    is_ambiguous_slash_date
)

date_audit = pd.DataFrame({
    "source_excel_row": employees_raw.index + 2,
    "employee_id": employees_raw["employee_id"],
    "hire_date_raw": employees_raw["hire_date"],
    "hire_date_parsed": hire_date_parsed
})

invalid_date_mask = hire_date_parsed.isna()
future_date_mask = hire_date_parsed > analysis_date

print("ПЕРЕВІРКА ДАТ НАЙМУ")
print(f"Некоректні дати: {invalid_date_mask.sum()}")
print(f"Неоднозначні дати зі слешем: {ambiguous_date_mask.sum()}")
print(f"Майбутні дати: {future_date_mask.sum()}")

print("\nНЕКОРЕКТНІ ДАТИ")
display(date_audit[invalid_date_mask])

print("\nМАЙБУТНІ ДАТИ")
display(date_audit[future_date_mask])

print("\nПРИКЛАДИ НЕОДНОЗНАЧНИХ ДАТ")
display(date_audit[ambiguous_date_mask].head(10))

ПЕРЕВІРКА ДАТ НАЙМУ
Некоректні дати: 3
Неоднозначні дати зі слешем: 37
Майбутні дати: 1

НЕКОРЕКТНІ ДАТИ


,source_excel_row,employee_id,hire_date_raw,hire_date_parsed
17,19,E1018,31/02/2024,NaT
76,78,E1077,not a date,NaT
150,152,E1151,not a date,NaT



МАЙБУТНІ ДАТИ


,source_excel_row,employee_id,hire_date_raw,hire_date_parsed
55,57,E1056,03/09/2026,2026-09-03



ПРИКЛАДИ НЕОДНОЗНАЧНИХ ДАТ


,source_excel_row,employee_id,hire_date_raw,hire_date_parsed
2,4,E1003,07/03/2024,2024-03-07
8,10,E1009,02/05/2024,2024-05-02
26,28,E1027,02/03/2018,2018-03-02
29,31,E1030,09/08/2021,2021-08-09
46,48,E1047,09/07/2022,2022-07-09
48,50,E1049,02/01/2021,2021-01-02
54,56,E1055,08/06/2018,2018-06-08
55,57,E1056,03/09/2026,2026-09-03
69,71,E1070,08/02/2016,2016-02-08
71,73,E1072,05/04/2025,2025-04-05


### 2.6 Reference Data and Department Consistency

In [ ]:
def create_mapping(reference_df, canonical_column, variants_column):
    """Створює словник: варіант написання → канонічне значення."""

    mapping = {}

    for _, row in reference_df.iterrows():
        canonical_value = row[canonical_column]
        variants = row[variants_column]

        if pd.isna(canonical_value):
            continue

        canonical_value = str(canonical_value).strip()

        # Додаємо до словника саме канонічне значення
        mapping[canonical_value.casefold()] = canonical_value

        # Додаємо всі допустимі варіанти з довідника
        if pd.notna(variants):
            for variant in str(variants).split(","):
                mapping[variant.strip().casefold()] = canonical_value

    return mapping


department_mapping = create_mapping(
    reference_tables,
    "canonical_department",
    "accepted_variants"
)

city_mapping = create_mapping(
    reference_tables,
    "canonical_city",
    "accepted_variants.1"
)

gender_mapping = create_mapping(
    reference_tables,
    "canonical_gender",
    "accepted_variants.2"
)

status_mapping = create_mapping(
    reference_tables,
    "canonical_status",
    "accepted_variants.3"
)


def map_to_canonical(value, mapping):
    """Повертає канонічне значення або NaN."""

    if pd.isna(value):
        return np.nan

    return mapping.get(str(value).strip().casefold(), np.nan)


department_canonical = employees_raw["department_name"].apply(
    lambda value: map_to_canonical(value, department_mapping)
)

city_canonical = employees_raw["city"].apply(
    lambda value: map_to_canonical(value, city_mapping)
)

gender_canonical = employees_raw["gender"].apply(
    lambda value: map_to_canonical(value, gender_mapping)
)

status_canonical = employees_raw["employment_status"].apply(
    lambda value: map_to_canonical(value, status_mapping)
)

print("НЕРОЗПІЗНАНІ НЕПОРОЖНІ ЗНАЧЕННЯ")
print(
    "department_name:",
    (employees_raw["department_name"].notna() & department_canonical.isna()).sum()
)
print(
    "city:",
    (employees_raw["city"].notna() & city_canonical.isna()).sum()
)
print(
    "gender:",
    (employees_raw["gender"].notna() & gender_canonical.isna()).sum()
)
print(
    "employment_status:",
    (
        employees_raw["employment_status"].notna()
        & status_canonical.isna()
    ).sum()
)

department_id_to_name = dict(
    zip(
        reference_tables["department_id"],
        reference_tables["canonical_department"]
    )
)

expected_department_name = employees_raw["department_id"].map(
    department_id_to_name
)

department_mismatch_mask = (
    expected_department_name.notna()
    & department_canonical.notna()
    & expected_department_name.ne(department_canonical)
)

print("\nПРОПУЩЕНІ DEPARTMENT_ID")
display(
    audit_data.loc[
        employees_raw["department_id"].isna(),
        [
            "source_excel_row",
            "employee_id",
            "department_id",
            "department_name"
        ]
    ]
)

print("\nНЕВІДПОВІДНОСТІ DEPARTMENT_ID І DEPARTMENT_NAME")
department_check = audit_data.loc[
    department_mismatch_mask,
    [
        "source_excel_row",
        "employee_id",
        "department_id",
        "department_name"
    ]
].copy()

department_check["expected_department_name"] = (
    expected_department_name[department_mismatch_mask]
)

display(department_check)

НЕРОЗПІЗНАНІ НЕПОРОЖНІ ЗНАЧЕННЯ
department_name: 0
city: 0
gender: 0
employment_status: 0

ПРОПУЩЕНІ DEPARTMENT_ID


,source_excel_row,employee_id,department_id,department_name
54,56,E1055,NaN,ops
131,133,E1132,NaN,Маркетинг



НЕВІДПОВІДНОСТІ DEPARTMENT_ID І DEPARTMENT_NAME


,source_excel_row,employee_id,department_id,department_name,expected_department_name
32,34,E1033,D08,sales,Legal
83,85,E1084,D03,Legl,Finance
159,161,E1160,D03,Tech,Finance


### 2.7 Performance Score Validation

In [ ]:
performance_numeric = pd.to_numeric(
    employees_raw["performance_score"],
    errors="coerce"
)

performance_audit = pd.DataFrame({
    "source_excel_row": employees_raw.index + 2,
    "employee_id": employees_raw["employee_id"],
    "performance_score_raw": employees_raw["performance_score"],
    "performance_score_numeric": performance_numeric
})

invalid_performance_mask = (
    performance_numeric.notna()
    & ~performance_numeric.between(1, 5)
)

print("ПЕРЕВІРКА PERFORMANCE_SCORE")
print(f"Пропущені значення: {performance_numeric.isna().sum()}")
print(
    "Значення поза допустимим діапазоном 1–5:",
    invalid_performance_mask.sum()
)

print("\nРОЗПОДІЛ ПОЧАТКОВИХ ЗНАЧЕНЬ")
display(
    performance_numeric
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("performance_score")
    .reset_index(name="count")
)

print("\nНЕКОРЕКТНІ ЗНАЧЕННЯ")
display(performance_audit[invalid_performance_mask])

ПЕРЕВІРКА PERFORMANCE_SCORE
Пропущені значення: 18
Значення поза допустимим діапазоном 1–5: 43

РОЗПОДІЛ ПОЧАТКОВИХ ЗНАЧЕНЬ


,performance_score,count
0,0.0,25
1,1.0,13
2,2.0,30
3,3.0,31
4,4.0,46
5,5.0,19
6,6.0,18
7,NaN,18



НЕКОРЕКТНІ ЗНАЧЕННЯ


,source_excel_row,employee_id,performance_score_raw,performance_score_numeric
0,2,E1001,6,6.0
2,4,E1003,0,0.0
6,8,E1007,6,6.0
9,11,E1010,6,6.0
10,12,E1011,6,6.0
13,15,E1014,0,0.0
17,19,E1018,0,0.0
28,30,E1029,0,0.0
34,36,E1035,0,0.0
37,39,E1038,0,0.0


## 3. Підготовка очищеного набору employees_cleaned

Оскільки Python було обрано основним інструментом очищення даних у Завданні 0, нижче виконується підготовка набору `employees_cleaned` за визначеними правилами: стандартизація текстових значень, перетворення типів даних, обробка пропусків і некоректних значень, видалення дублікатів та позначення потенційних викидів зарплати.

 Вихідний DataFrame `employees_raw` залишається незмінним. Спочатку видаляємо підтверджений повний дублікат і виправляємо некоректні та повторні `employee_id`.

In [ ]:
# Створюємо окрему таблицю та видаляємо лише повний дублікат
employees_cleaned = employees_raw.drop_duplicates(keep="first").copy()

# Зберігаємо номер початкового рядка Excel
employees_cleaned["source_excel_row"] = employees_cleaned.index + 2

# Зберігаємо початковий ID для аудиту змін
employees_cleaned["original_employee_id"] = (
    employees_cleaned["employee_id"]
)

# Прибираємо пробіли в ID
employees_cleaned["employee_id"] = (
    employees_cleaned["employee_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Виправлення ID, обґрунтовані послідовністю записів
employee_id_corrections = {
    46: "E1045",
    47: "E1046",
    103: "E1102",
    158: "E1157",
    189: "E1188"
}

employees_cleaned["employee_id"] = (
    employees_cleaned["source_excel_row"]
    .map(employee_id_corrections)
    .fillna(employees_cleaned["employee_id"])
)

# Прибираємо зайві пробіли в іменах і назвах посад
for column in ["full_name", "job_title"]:
    employees_cleaned[column] = (
        employees_cleaned[column]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Оновлюємо індекс після видалення дубліката
employees_cleaned = employees_cleaned.reset_index(drop=True)

print("ПЕРЕВІРКА ПІСЛЯ ОЧИЩЕННЯ EMPLOYEE_ID")
print(f"Кількість рядків: {len(employees_cleaned)}")
print(
    "Повторні employee_id:",
    employees_cleaned["employee_id"].duplicated().sum()
)
print(
    "ID неправильного формату:",
    (
        ~employees_cleaned["employee_id"]
        .str.match(r"^E\d{4}$")
    ).sum()
)

display(
    employees_cleaned.loc[
        employees_cleaned["original_employee_id"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne(employees_cleaned["employee_id"]),
        [
            "source_excel_row",
            "original_employee_id",
            "employee_id",
            "full_name"
        ]
    ]
)

ПЕРЕВІРКА ПІСЛЯ ОЧИЩЕННЯ EMPLOYEE_ID
Кількість рядків: 199
Повторні employee_id: 0
ID неправильного формату: 0


,source_excel_row,original_employee_id,employee_id,full_name
44,46,E1010,E1045,Mykhailo Kravchenko
45,47,E1010,E1046,Ivan Kozak
101,103,E10A2,E1102,Taras Kozak
156,158,NaN,E1157,Taras Marchenko
187,189,,E1188,Oleksandr Savchenko


In [ ]:
# Зберігаємо початкові категоріальні значення для аудиту
for column in [
    "department_id",
    "department_name",
    "city",
    "gender",
    "employment_status"
]:
    employees_cleaned[f"original_{column}"] = employees_cleaned[column]


# Канонічні назви відділів за текстовим довідником
department_from_name = employees_cleaned["department_name"].apply(
    lambda value: map_to_canonical(value, department_mapping)
)

# Відновлюємо пропущені department_id за канонічною назвою
department_name_to_id = {
    department_name: department_id
    for department_id, department_name
    in department_id_to_name.items()
}

employees_cleaned["department_id"] = (
    employees_cleaned["department_id"]
    .fillna(department_from_name.map(department_name_to_id))
)

# Назву відділу визначаємо за department_id
employees_cleaned["department_name"] = (
    employees_cleaned["department_id"]
    .map(department_id_to_name)
)

# Стандартизуємо інші категоріальні поля
employees_cleaned["city"] = (
    employees_cleaned["city"]
    .apply(lambda value: map_to_canonical(value, city_mapping))
)

employees_cleaned["gender"] = (
    employees_cleaned["gender"]
    .apply(lambda value: map_to_canonical(value, gender_mapping))
    .fillna("Not specified")
)

employees_cleaned["employment_status"] = (
    employees_cleaned["employment_status"]
    .apply(lambda value: map_to_canonical(value, status_mapping))
)

# Ознака активного співробітника
employees_cleaned["is_active"] = (
    employees_cleaned["employment_status"]
    .eq("Active")
    .astype(int)
)

print("ПЕРЕВІРКА КАТЕГОРІАЛЬНИХ ПОЛІВ")

for column in [
    "department_name",
    "city",
    "gender",
    "employment_status"
]:
    print(f"\n{column}")
    print(
        employees_cleaned[column]
        .value_counts(dropna=False)
        .to_dict()
    )

print("\nПРОПУЩЕНІ DEPARTMENT_ID")
print(employees_cleaned["department_id"].isna().sum())

print("\nНЕВІДПОВІДНОСТІ ID І НАЗВИ ВІДДІЛУ")
print(
    (
        employees_cleaned["department_name"]
        != employees_cleaned["department_id"].map(
            department_id_to_name
        )
    ).sum()
)

print("\nКІЛЬКІСТЬ АКТИВНИХ СПІВРОБІТНИКІВ")
print(employees_cleaned["is_active"].sum())

ПЕРЕВІРКА КАТЕГОРІАЛЬНИХ ПОЛІВ

department_name
{'IT': 36, 'HR': 28, 'Legal': 27, 'Customer Support': 26, 'Operations': 25, 'Sales': 22, 'Marketing': 19, 'Finance': 16}

city
{'Remote': 37, 'Kyiv': 36, 'Dnipro': 36, 'Lviv': 32, 'Odesa': 32, 'Kharkiv': 26}

gender
{'Male': 74, 'Female': 67, 'Not specified': 58}

employment_status
{'Active': 156, 'Terminated': 31, 'On leave': 12}

ПРОПУЩЕНІ DEPARTMENT_ID
0

НЕВІДПОВІДНОСТІ ID І НАЗВИ ВІДДІЛУ
0

КІЛЬКІСТЬ АКТИВНИХ СПІВРОБІТНИКІВ
156


In [ ]:
# Зберігаємо початкове значення
employees_cleaned["original_salary"] = employees_cleaned["salary"]

# Перетворюємо зарплату на число
employees_cleaned["salary"] = (
    employees_cleaned["salary"]
    .apply(parse_salary)
)

# Межі IQR після видалення дубліката
positive_salary_cleaned = employees_cleaned.loc[
    employees_cleaned["salary"] > 0,
    "salary"
]

salary_q1 = positive_salary_cleaned.quantile(0.25)
salary_q3 = positive_salary_cleaned.quantile(0.75)
salary_iqr = salary_q3 - salary_q1

salary_lower_bound = salary_q1 - 1.5 * salary_iqr
salary_upper_bound = salary_q3 + 1.5 * salary_iqr

# Статистичними викидами вважаємо лише додатні зарплати,
# які виходять за межі IQR
employees_cleaned["salary_outlier_flag"] = (
    employees_cleaned["salary"].gt(0)
    & (
        (employees_cleaned["salary"] < salary_lower_bound)
        | (employees_cleaned["salary"] > salary_upper_bound)
    )
).astype(int)

# Екстремальний позитивний викид:
# зарплата більш ніж удвічі перевищує верхню межу IQR
extreme_salary_mask = (
    employees_cleaned["salary"]
    > 2 * salary_upper_bound
)

# Некоректні зарплати: пропуск, нуль або від’ємне значення
invalid_salary_mask = (
    employees_cleaned["salary"].isna()
    | (employees_cleaned["salary"] <= 0)
)

# Позначаємо всі значення, виключені з основної аналітики
employees_cleaned["salary_excluded_flag"] = (
    invalid_salary_mask | extreme_salary_mask
).astype(int)

# Виключені значення замінюємо на NaN
employees_cleaned.loc[
    invalid_salary_mask | extreme_salary_mask,
    "salary"
] = np.nan

print("ПЕРЕВІРКА ОЧИЩЕННЯ ЗАРПЛАТ")
print(f"Q1: {salary_q1:,.0f}")
print(f"Q3: {salary_q3:,.0f}")
print(f"Верхня межа IQR: {salary_upper_bound:,.0f}")
print(
    "Межа екстремального викиду:",
    f"{2 * salary_upper_bound:,.0f}"
)
print(
    "Потенційні статистичні викиди:",
    employees_cleaned["salary_outlier_flag"].sum()
)
print(
    "Значення, виключені з основної аналітики:",
    employees_cleaned["salary_excluded_flag"].sum()
)
print(
    "Пропуски в очищеній salary:",
    employees_cleaned["salary"].isna().sum()
)

print("\nВИКЛЮЧЕНІ ЗНАЧЕННЯ")
display(
    employees_cleaned.loc[
        employees_cleaned["salary_excluded_flag"] == 1,
        [
            "source_excel_row",
            "employee_id",
            "job_title",
            "original_salary",
            "salary",
            "salary_outlier_flag",
            "salary_excluded_flag"
        ]
    ]
)

ПЕРЕВІРКА ОЧИЩЕННЯ ЗАРПЛАТ
Q1: 37,000
Q3: 57,000
Верхня межа IQR: 87,000
Межа екстремального викиду: 174,000
Потенційні статистичні викиди: 6
Значення, виключені з основної аналітики: 8
Пропуски в очищеній salary: 8

ВИКЛЮЧЕНІ ЗНАЧЕННЯ


,source_excel_row,employee_id,job_title,original_salary,salary,salary_outlier_flag,salary_excluded_flag
11,13,E1012,Support Team Lead,0,NaN,0,1
28,30,E1029,Sales Representative,NaN,NaN,0,1
42,44,E1043,PPC Specialist,-40506,NaN,0,1
65,67,E1066,Onboarding Specialist,0,NaN,0,1
94,96,E1095,Recruiter,250000,NaN,1,1
117,119,E1118,Sales Representative,NaN,NaN,0,1
140,142,E1141,Compliance Specialist,-42414,NaN,0,1
172,174,E1173,Data Analyst,300 000 грн,NaN,1,1


In [ ]:
# Зберігаємо початкову дату
employees_cleaned["original_hire_date"] = (
    employees_cleaned["hire_date"]
)

# Розпізнаємо дату за раніше визначеними правилами
employees_cleaned["hire_date"] = (
    employees_cleaned["hire_date"]
    .apply(parse_hire_date)
)

# Позначаємо некоректні та майбутні дати
invalid_or_future_date_mask = (
    employees_cleaned["hire_date"].isna()
    | (employees_cleaned["hire_date"] > analysis_date)
)

employees_cleaned["hire_date_issue_flag"] = (
    invalid_or_future_date_mask.astype(int)
)

# Майбутні дати також перетворюємо на NaT
employees_cleaned.loc[
    employees_cleaned["hire_date"] > analysis_date,
    "hire_date"
] = pd.NaT


def calculate_full_months(hire_date, end_date):
    """Розраховує стаж у повних місяцях."""

    if pd.isna(hire_date):
        return np.nan

    months = (
        (end_date.year - hire_date.year) * 12
        + end_date.month
        - hire_date.month
    )

    # Якщо день місяця ще не настав, останній місяць неповний
    if end_date.day < hire_date.day:
        months -= 1

    return months


employees_cleaned["tenure_months"] = (
    employees_cleaned["hire_date"]
    .apply(
        lambda date: calculate_full_months(
            date,
            analysis_date
        )
    )
    .astype("Int64")
)

print("ПЕРЕВІРКА ОЧИЩЕННЯ ДАТ")
print(
    "Пропуски в очищеній hire_date:",
    employees_cleaned["hire_date"].isna().sum()
)
print(
    "Майбутні дати після очищення:",
    (employees_cleaned["hire_date"] > analysis_date).sum()
)
print(
    "Пропуски в tenure_months:",
    employees_cleaned["tenure_months"].isna().sum()
)

print("\nПРОБЛЕМНІ ДАТИ")
display(
    employees_cleaned.loc[
        employees_cleaned["hire_date_issue_flag"] == 1,
        [
            "source_excel_row",
            "employee_id",
            "original_hire_date",
            "hire_date",
            "tenure_months",
            "hire_date_issue_flag"
        ]
    ]
)

ПЕРЕВІРКА ОЧИЩЕННЯ ДАТ
Пропуски в очищеній hire_date: 4
Майбутні дати після очищення: 0
Пропуски в tenure_months: 4

ПРОБЛЕМНІ ДАТИ


,source_excel_row,employee_id,original_hire_date,hire_date,tenure_months,hire_date_issue_flag
17,19,E1018,31/02/2024,NaT,<NA>,1
55,57,E1056,03/09/2026,NaT,<NA>,1
76,78,E1077,not a date,NaT,<NA>,1
150,152,E1151,not a date,NaT,<NA>,1


In [ ]:
# Зберігаємо початкове значення
employees_cleaned["original_performance_score"] = (
    employees_cleaned["performance_score"]
)

# Перетворюємо оцінку на число
employees_cleaned["performance_score"] = pd.to_numeric(
    employees_cleaned["performance_score"],
    errors="coerce"
)

# Виявляємо значення поза шкалою 1–5
invalid_performance_cleaned_mask = (
    employees_cleaned["performance_score"].notna()
    & ~employees_cleaned["performance_score"].between(1, 5)
)

employees_cleaned["performance_score_issue_flag"] = (
    invalid_performance_cleaned_mask.astype(int)
)

# Некоректні значення замінюємо на NaN
employees_cleaned.loc[
    invalid_performance_cleaned_mask,
    "performance_score"
] = np.nan

# Використовуємо nullable integer
employees_cleaned["performance_score"] = (
    employees_cleaned["performance_score"]
    .astype("Int64")
)

print("ПЕРЕВІРКА PERFORMANCE_SCORE ПІСЛЯ ОЧИЩЕННЯ")
print(
    "Некоректні значення, замінені на пропуски:",
    employees_cleaned["performance_score_issue_flag"].sum()
)
print(
    "Усього пропусків після очищення:",
    employees_cleaned["performance_score"].isna().sum()
)
print(
    "Значення поза діапазоном після очищення:",
    (
        employees_cleaned["performance_score"].notna()
        & ~employees_cleaned["performance_score"].between(1, 5)
    ).sum()
)

print("\nРОЗПОДІЛ ОЧИЩЕНИХ ОЦІНОК")
display(
    employees_cleaned["performance_score"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("performance_score")
    .reset_index(name="count")
)

ПЕРЕВІРКА PERFORMANCE_SCORE ПІСЛЯ ОЧИЩЕННЯ
Некоректні значення, замінені на пропуски: 43
Усього пропусків після очищення: 61
Значення поза діапазоном після очищення: 0

РОЗПОДІЛ ОЧИЩЕНИХ ОЦІНОК


,performance_score,count
0,1,13
1,2,30
2,3,31
3,4,46
4,5,18
5,<NA>,61


In [ ]:
# Зарплати активних працівників, придатні для аналізу
active_valid_salary = employees_cleaned.loc[
    (employees_cleaned["is_active"] == 1)
    & employees_cleaned["salary"].notna(),
    "salary"
]

active_salary_q1 = active_valid_salary.quantile(0.25)
active_salary_median = active_valid_salary.median()
active_salary_q3 = active_valid_salary.quantile(0.75)

print("ПОРОГИ ДЛЯ ДОДАТКОВИХ КАТЕГОРІЙ")
print(f"Q1 активних працівників: {active_salary_q1:,.0f}")
print(
    "Медіана активних працівників:",
    f"{active_salary_median:,.0f}"
)
print(f"Q3 активних працівників: {active_salary_q3:,.0f}")


def classify_salary_level(salary):
    """Класифікує зарплату за квартилями активних працівників."""

    if pd.isna(salary):
        return "Unknown"
    elif salary <= active_salary_q1:
        return "Low"
    elif salary <= active_salary_q3:
        return "Medium"
    else:
        return "High"


def compare_with_median(salary):
    """Порівнює зарплату з медіаною активних працівників."""

    if pd.isna(salary):
        return "Unknown"
    elif salary < active_salary_median:
        return "Below median"
    elif salary > active_salary_median:
        return "Above median"
    else:
        return "Equal to median"


employees_cleaned["salary_level"] = (
    employees_cleaned["salary"]
    .apply(classify_salary_level)
)

employees_cleaned["salary_vs_median"] = (
    employees_cleaned["salary"]
    .apply(compare_with_median)
)

print("\nРОЗПОДІЛ РІВНІВ ЗАРПЛАТИ")
display(
    employees_cleaned["salary_level"]
    .value_counts(dropna=False)
    .rename_axis("salary_level")
    .reset_index(name="count")
)

print("\nПОРІВНЯННЯ З МЕДІАНОЮ")
display(
    employees_cleaned["salary_vs_median"]
    .value_counts(dropna=False)
    .rename_axis("salary_vs_median")
    .reset_index(name="count")
)

ПОРОГИ ДЛЯ ДОДАТКОВИХ КАТЕГОРІЙ
Q1 активних працівників: 36,250
Медіана активних працівників: 45,000
Q3 активних працівників: 58,750

РОЗПОДІЛ РІВНІВ ЗАРПЛАТИ


,salary_level,count
0,Medium,98
1,Low,48
2,High,45
3,Unknown,8



ПОРІВНЯННЯ З МЕДІАНОЮ


,salary_vs_median,count
0,Above median,96
1,Below median,92
2,Unknown,8
3,Equal to median,3


In [ ]:
print("ФІНАЛЬНА ПЕРЕВІРКА EMPLOYEES_CLEANED")

quality_checks = {
    "Кількість рядків": len(employees_cleaned),
    "Повні дублікати": employees_cleaned[
        [
            "employee_id",
            "full_name",
            "department_id",
            "department_name",
            "job_title",
            "salary",
            "hire_date",
            "gender",
            "city",
            "employment_status",
            "performance_score"
        ]
    ].duplicated().sum(),
    "Повторні employee_id": (
        employees_cleaned["employee_id"].duplicated().sum()
    ),
    "Некоректні employee_id": (
        ~employees_cleaned["employee_id"]
        .str.match(r"^E\d{4}$")
    ).sum(),
    "Пропущені department_id": (
        employees_cleaned["department_id"].isna().sum()
    ),
    "Пропущені department_name": (
        employees_cleaned["department_name"].isna().sum()
    ),
    "Пропущені city": (
        employees_cleaned["city"].isna().sum()
    ),
    "Пропущені gender": (
        employees_cleaned["gender"].isna().sum()
    ),
    "Пропущені employment_status": (
        employees_cleaned["employment_status"].isna().sum()
    ),
    "Невалідні зарплати після очищення": (
        (
            employees_cleaned["salary"].notna()
            & (employees_cleaned["salary"] <= 0)
        ).sum()
    ),
    "Пропущені salary": (
        employees_cleaned["salary"].isna().sum()
    ),
    "Майбутні hire_date": (
        (
            employees_cleaned["hire_date"].notna()
            & (employees_cleaned["hire_date"] > analysis_date)
        ).sum()
    ),
    "Пропущені hire_date": (
        employees_cleaned["hire_date"].isna().sum()
    ),
    "Performance score поза 1–5": (
        (
            employees_cleaned["performance_score"].notna()
            & ~employees_cleaned["performance_score"].between(1, 5)
        ).sum()
    ),
    "Пропущені performance_score": (
        employees_cleaned["performance_score"].isna().sum()
    )
}

quality_check_table = pd.DataFrame(
    quality_checks.items(),
    columns=["check", "result"]
)

display(quality_check_table)

print("\nТИПИ ОСНОВНИХ ОЧИЩЕНИХ ПОЛІВ")
display(
    employees_cleaned[
        [
            "employee_id",
            "department_id",
            "salary",
            "hire_date",
            "gender",
            "city",
            "employment_status",
            "performance_score",
            "tenure_months",
            "is_active"
        ]
    ].dtypes.to_frame(name="data_type")
)

print("\nПЕРШІ 5 РЯДКІВ ОЧИЩЕНОЇ ТАБЛИЦІ")
display(employees_cleaned.head())

ФІНАЛЬНА ПЕРЕВІРКА EMPLOYEES_CLEANED


,check,result
0,Кількість рядків,199
1,Повні дублікати,0
2,Повторні employee_id,0
3,Некоректні employee_id,0
4,Пропущені department_id,0
5,Пропущені department_name,0
6,Пропущені city,0
7,Пропущені gender,0
8,Пропущені employment_status,0
9,Невалідні зарплати після очищення,0



ТИПИ ОСНОВНИХ ОЧИЩЕНИХ ПОЛІВ


,data_type
employee_id,object
department_id,object
salary,float64
hire_date,datetime64[ns]
gender,object
city,object
employment_status,object
performance_score,Int64
tenure_months,Int64
is_active,int64



ПЕРШІ 5 РЯДКІВ ОЧИЩЕНОЇ ТАБЛИЦІ


,employee_id,full_name,department_id,department_name,job_title,salary,hire_date,gender,city,employment_status,...,original_salary,salary_outlier_flag,salary_excluded_flag,original_hire_date,hire_date_issue_flag,tenure_months,original_performance_score,performance_score_issue_flag,salary_level,salary_vs_median
0,E1001,Iryna Lysenko,D06,Operations,Procurement Specialist,32000.0,2016-05-10,Female,Lviv,Active,...,32000,0,0,2016-05-10,0,123,6,1,Low,Below median
1,E1002,Yurii Rudenko,D05,IT,Software Engineer,91000.0,2017-10-15,Not specified,Remote,Active,...,91000,1,0,15/10/2017,0,106,5,0,High,Above median
2,E1003,Oleksandr Kravchenko,D02,Marketing,Marketing Specialist,48000.0,2024-03-07,Male,Kyiv,Terminated,...,"48,000",0,0,07/03/2024,0,29,0,1,Medium,Above median
3,E1004,Anastasiia Koval,D01,Sales,Sales Representative,29000.0,2025-08-04,Female,Kharkiv,On leave,...,"29,000",0,0,04.08.2025,0,12,4,0,Low,Below median
4,E1005,Maksym Moroz,D05,IT,Software Engineer,69000.0,2021-12-27,Male,Odesa,Active,...,69000,0,0,27.12.2021,0,56,2,0,High,Above median


## 4. Summary Metrics

Розраховуємо підсумкові показники для активних працівників із валідною зарплатою за відділами, містами, статтю та посадами.

In [ ]:
# Для зарплатних агрегацій використовуємо активних працівників
# із валідними зарплатами, не виключеними з аналізу
analysis_data = employees_cleaned.loc[
    (employees_cleaned["is_active"] == 1)
    & (employees_cleaned["salary_excluded_flag"] == 0)
    & employees_cleaned["salary"].notna()
].copy()

print("ДАНІ ДЛЯ АГРЕГАЦІЙ")
print(f"Кількість працівників: {len(analysis_data)}")
print(f"Загальний місячний ФОП: {analysis_data['salary'].sum():,.0f}")
print(f"Середня зарплата: {analysis_data['salary'].mean():,.0f}")
print(f"Медіанна зарплата: {analysis_data['salary'].median():,.0f}")

ДАНІ ДЛЯ АГРЕГАЦІЙ
Кількість працівників: 150
Загальний місячний ФОП: 7,290,000
Середня зарплата: 48,600
Медіанна зарплата: 45,000


In [ ]:
def calculate_summary(data, group_column):
    """Розраховує підсумкові показники за вибраною категорією."""

    summary = (
        data.groupby(group_column, dropna=False)
        .agg(
            employee_count=("employee_id", "nunique"),
            average_salary=("salary", "mean"),
            median_salary=("salary", "median"),
            minimum_salary=("salary", "min"),
            maximum_salary=("salary", "max"),
            total_payroll=("salary", "sum"),
            average_performance=("performance_score", "mean"),
            average_tenure_months=("tenure_months", "mean")
        )
        .reset_index()
    )

    numeric_columns = [
        "average_salary",
        "median_salary",
        "minimum_salary",
        "maximum_salary",
        "total_payroll",
        "average_performance",
        "average_tenure_months"
    ]

    summary[numeric_columns] = summary[numeric_columns].round(2)

    return summary.sort_values(
        "total_payroll",
        ascending=False
    ).reset_index(drop=True)

In [ ]:
summary_by_department = calculate_summary(
    analysis_data,
    "department_name"
)

summary_by_city = calculate_summary(
    analysis_data,
    "city"
)

summary_by_gender = calculate_summary(
    analysis_data,
    "gender"
)

summary_by_job_title = calculate_summary(
    analysis_data,
    "job_title"
)

In [ ]:
print("ПІДСУМКОВІ ПОКАЗНИКИ ПО ВІДДІЛАХ")
display(summary_by_department)

print("ПІДСУМКОВІ ПОКАЗНИКИ ПО МІСТАХ")
display(summary_by_city)

print("ПІДСУМКОВІ ПОКАЗНИКИ ЗА СТАТТЮ")
display(summary_by_gender)

print("ПІДСУМКОВІ ПОКАЗНИКИ ПО ПОСАДАХ")
display(summary_by_job_title)

ПІДСУМКОВІ ПОКАЗНИКИ ПО ВІДДІЛАХ


,department_name,employee_count,average_salary,median_salary,minimum_salary,maximum_salary,total_payroll,average_performance,average_tenure_months
0,IT,26,63653.85,60000.0,41000.0,102000.0,1655000.0,3.09,69.54
1,Legal,21,52285.71,48000.0,32000.0,90000.0,1098000.0,3.36,66.5
2,HR,22,42454.55,40500.0,26000.0,62000.0,934000.0,3.11,66.23
3,Operations,22,41000.00,38000.0,26000.0,72000.0,902000.0,3.75,52.15
4,Finance,15,55266.67,54000.0,29000.0,88000.0,829000.0,3.29,51.53
5,Customer Support,18,44388.89,42000.0,29000.0,69000.0,799000.0,3.73,39.11
6,Sales,14,44357.14,41500.0,26000.0,82000.0,621000.0,3.2,83.31
7,Marketing,12,37666.67,36000.0,27000.0,54000.0,452000.0,3.7,66.17


ПІДСУМКОВІ ПОКАЗНИКИ ПО МІСТАХ


,city,employee_count,average_salary,median_salary,minimum_salary,maximum_salary,total_payroll,average_performance,average_tenure_months
0,Kyiv,28,53607.14,52500.0,29000.0,88000.0,1501000.0,3.33,53.07
1,Remote,27,53259.26,46000.0,26000.0,91000.0,1438000.0,3.43,61.44
2,Odesa,24,54625.00,51000.0,28000.0,102000.0,1311000.0,3.44,55.46
3,Dnipro,29,42103.45,38000.0,26000.0,76000.0,1221000.0,3.18,64.1
4,Lviv,23,43913.04,41000.0,28000.0,63000.0,1010000.0,3.0,79.57
5,Kharkiv,19,42578.95,43000.0,26000.0,62000.0,809000.0,3.77,58.37


ПІДСУМКОВІ ПОКАЗНИКИ ЗА СТАТТЮ


,gender,employee_count,average_salary,median_salary,minimum_salary,maximum_salary,total_payroll,average_performance,average_tenure_months
0,Male,56,47892.86,47500.0,26000.0,82000.0,2682000.0,3.36,62.0
1,Not specified,45,51933.33,45000.0,27000.0,102000.0,2337000.0,3.31,62.93
2,Female,49,46346.94,43000.0,26000.0,88000.0,2271000.0,3.37,59.92


ПІДСУМКОВІ ПОКАЗНИКИ ПО ПОСАДАХ


,job_title,employee_count,average_salary,median_salary,minimum_salary,maximum_salary,total_payroll,average_performance,average_tenure_months
0,Software Engineer,9,77111.11,74000.0,60000.0,102000.0,694000.0,3.25,69.78
1,QA Engineer,11,57636.36,59000.0,41000.0,79000.0,634000.0,3.25,72.91
2,Contract Manager,8,51500.00,50500.0,41000.0,63000.0,412000.0,3.25,69.14
3,People Partner,7,53000.00,55000.0,43000.0,62000.0,371000.0,3.0,52.57
4,Legal Counsel,5,72200.00,69000.0,58000.0,90000.0,361000.0,3.0,42.6
5,Finance Manager,4,79000.00,79000.0,70000.0,88000.0,316000.0,2.67,57.25
6,Operations Manager,5,59200.00,56000.0,50000.0,72000.0,296000.0,4.25,41.8
7,Customer Success Manager,5,54800.00,54000.0,42000.0,69000.0,274000.0,5.0,41.2
8,System Administrator,5,51200.00,51000.0,47000.0,56000.0,256000.0,2.6,64.6
9,Procurement Specialist,6,38833.33,40000.0,28000.0,49000.0,233000.0,4.0,55.0


## 5. Salary Anomalies and Comparison with Job Average

### 5.1 Salary Outliers

Потенційні викиди зарплати визначаються методом IQR. Викидом вважається додатне значення, яке виходить за межі:

- нижня межа: Q1 − 1.5 × IQR;
- верхня межа: Q3 + 1.5 × IQR.

In [ ]:
# Отримуємо числову зарплату з початкового значення,
# оскільки екстремальні значення могли бути замінені на NaN
employees_cleaned["salary_for_outlier_analysis"] = (
    employees_cleaned["original_salary"]
    .apply(parse_salary)
)

salary_outliers = (
    employees_cleaned.loc[
        employees_cleaned["salary_outlier_flag"] == 1,
        [
            "employee_id",
            "full_name",
            "department_name",
            "job_title",
            "salary_for_outlier_analysis",
            "employment_status",
            "salary_outlier_flag",
            "salary_excluded_flag"
        ]
    ]
    .sort_values(
        "salary_for_outlier_analysis",
        ascending=False
    )
    .reset_index(drop=True)
)

print("АНОМАЛЬНІ ЗНАЧЕННЯ ЗАРПЛАТИ")
print(f"Нижня межа IQR: {salary_lower_bound:,.0f}")
print(f"Верхня межа IQR: {salary_upper_bound:,.0f}")
print(f"Кількість знайдених викидів: {len(salary_outliers)}")

display(salary_outliers)

АНОМАЛЬНІ ЗНАЧЕННЯ ЗАРПЛАТИ
Нижня межа IQR: 7,000
Верхня межа IQR: 87,000
Кількість знайдених викидів: 6


,employee_id,full_name,department_name,job_title,salary_for_outlier_analysis,employment_status,salary_outlier_flag,salary_excluded_flag
0,E1173,Nadiia Tkachenko,IT,Data Analyst,300000.0,Active,1,1
1,E1095,Ivan Kravchenko,HR,Recruiter,250000.0,Active,1,1
2,E1130,Viktoriia Lysenko,IT,Software Engineer,102000.0,Active,1,0
3,E1002,Yurii Rudenko,IT,Software Engineer,91000.0,Active,1,0
4,E1198,Tetiana Lysenko,Legal,Legal Counsel,90000.0,Active,1,0
5,E1196,Kateryna Marchenko,Finance,Finance Manager,88000.0,Active,1,0


### 5.2 Employees Below the Average Salary for Their Job

Для кожної посади розраховується середня зарплата активних працівників із валідними зарплатами. Після цього зарплата кожного працівника порівнюється із середнім показником для його посади.

In [ ]:
# Середня зарплата за посадою серед активних працівників
# із валідними зарплатами
analysis_data["average_salary_by_job"] = (
    analysis_data.groupby("job_title")["salary"]
    .transform("mean")
)

# Різниця між зарплатою працівника та середньою по посаді
analysis_data["difference_from_job_average"] = (
    analysis_data["salary"]
    - analysis_data["average_salary_by_job"]
)

analysis_data["difference_percent"] = (
    analysis_data["difference_from_job_average"]
    / analysis_data["average_salary_by_job"]
    * 100
)

employees_below_job_average = (
    analysis_data.loc[
        analysis_data["salary"]
        < analysis_data["average_salary_by_job"],
        [
            "employee_id",
            "full_name",
            "department_name",
            "job_title",
            "salary",
            "average_salary_by_job",
            "difference_from_job_average",
            "difference_percent"
        ]
    ]
    .copy()
)

columns_to_round = [
    "average_salary_by_job",
    "difference_from_job_average",
    "difference_percent"
]

employees_below_job_average[columns_to_round] = (
    employees_below_job_average[columns_to_round]
    .round(2)
)

employees_below_job_average = (
    employees_below_job_average
    .sort_values(
        ["job_title", "salary"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

print("ПРАЦІВНИКИ ІЗ ЗАРПЛАТОЮ НИЖЧЕ СЕРЕДНЬОЇ ПО ПОСАДІ")
print(
    "Кількість працівників:",
    len(employees_below_job_average)
)

display(employees_below_job_average)

ПРАЦІВНИКИ ІЗ ЗАРПЛАТОЮ НИЖЧЕ СЕРЕДНЬОЇ ПО ПОСАДІ
Кількість працівників: 74


,employee_id,full_name,department_name,job_title,salary,average_salary_by_job,difference_from_job_average,difference_percent
0,E1105,Dmytro Bondarenko,Sales,Account Executive,39000.0,42500.00,-3500.00,-8.24
1,E1062,Tetiana Rudenko,Sales,Account Executive,40000.0,42500.00,-2500.00,-5.88
2,E1166,Mykhailo Moroz,Finance,Accountant,34000.0,44000.00,-10000.00,-22.73
3,E1171,Andrii Moroz,Sales,Business Development Manager,46000.0,64666.67,-18666.67,-28.87
4,E1051,Dmytro Melnyk,Legal,Compliance Specialist,38000.0,56333.33,-18333.33,-32.54
...,...,...,...,...,...,...,...,...
69,E1175,Viktoriia Kozak,IT,System Administrator,51000.0,51200.00,-200.00,-0.39
70,E1189,Anastasiia Tkachenko,Customer Support,Technical Support Specialist,32000.0,38000.00,-6000.00,-15.79
71,E1054,Taras Polishchuk,Customer Support,Technical Support Specialist,34000.0,38000.00,-4000.00,-10.53
72,E1036,Maksym Savchenko,Customer Support,Technical Support Specialist,37000.0,38000.00,-1000.00,-2.63


## Cleaning Report

У звіті зафіксовано знайдені проблеми, правила їх обробки, кількість затронутих рядків і зроблені припущення. Після очищення залишилося 199 рядків.

In [ ]:
def count_changed_values(original_series, cleaned_series):
    """Рахує кількість значень, змінених під час стандартизації."""

    original_normalized = (
        original_series
        .fillna("<MISSING>")
        .astype(str)
        .str.strip()
    )

    cleaned_normalized = (
        cleaned_series
        .fillna("<MISSING>")
        .astype(str)
        .str.strip()
    )

    return original_normalized.ne(cleaned_normalized).sum()


department_name_changes = count_changed_values(
    employees_cleaned["original_department_name"],
    employees_cleaned["department_name"]
)

city_changes = count_changed_values(
    employees_cleaned["original_city"],
    employees_cleaned["city"]
)

gender_changes = count_changed_values(
    employees_cleaned["original_gender"],
    employees_cleaned["gender"]
)

status_changes = count_changed_values(
    employees_cleaned["original_employment_status"],
    employees_cleaned["employment_status"]
)


cleaning_report = pd.DataFrame([
    {
        "issue_id": 1,
        "column": "employee_id",
        "problem_type": "duplicates / invalid ID",
        "description/example": (
            "1 exact duplicate; repeated E1010; invalid E10A2; "
            "one missing ID and one whitespace-only ID"
        ),
        "decision/rule": (
            "Removed the exact duplicate and restored 5 IDs "
            "from the sequential numbering pattern"
        ),
        "rows_affected": 6,
        "assumption/comment": (
            "Corrections were recorded in original_employee_id. "
            "199 rows remained after removing one exact duplicate."
        )
    },
    {
        "issue_id": 2,
        "column": "department_name",
        "problem_type": "inconsistent spelling",
        "description/example": (
            "40 raw spellings, including MKT, Marketng, Tech, "
            "Legl and Ukrainian variants"
        ),
        "decision/rule": (
            "Mapped all values to 8 canonical departments "
            "using reference_tables"
        ),
        "rows_affected": int(department_name_changes),
        "assumption/comment": (
            "Canonical department names from reference_tables "
            "were treated as authoritative."
        )
    },
    {
        "issue_id": 3,
        "column": "department_id vs department_name",
        "problem_type": "mismatch",
        "description/example": (
            "2 missing department_id values and 3 conflicts "
            "between ID and department name"
        ),
        "decision/rule": (
            "Recovered missing IDs from canonical names; "
            "for conflicts, department_id had priority"
        ),
        "rows_affected": 5,
        "assumption/comment": (
            "department_id was treated as the primary structured key."
        )
    },
    {
        "issue_id": 4,
        "column": "salary",
        "problem_type": (
            "text format / zero / negative / missing / outlier"
        ),
        "description/example": (
            "Currency text formats, 2 missing, 2 zero, "
            "2 negative and 2 extreme positive values"
        ),
        "decision/rule": (
            "Converted salary to numeric; excluded missing, "
            "non-positive and extreme values above 2× the IQR "
            "upper bound"
        ),
        "rows_affected": int(
            employees_cleaned["salary_excluded_flag"].sum()
        ),
        "assumption/comment": (
            "IQR outliers were flagged, not removed automatically. "
            "Plausible salaries 88,000–102,000 were retained."
        )
    },
    {
        "issue_id": 5,
        "column": "hire_date",
        "problem_type": (
            "mixed date formats / invalid / future date"
        ),
        "description/example": (
            "Mixed ISO, dot and slash formats; 37 ambiguous slash "
            "dates; 3 invalid and 1 future date"
        ),
        "decision/rule": (
            "Parsed dates by explicit rules; ambiguous slash dates "
            "were treated as dd/mm/yyyy; invalid and future dates "
            "were converted to NaT"
        ),
        "rows_affected": 40,
        "assumption/comment": (
            "Analysis date fixed at 2026-09-01. European format "
            "was selected because the dataset uses UAH."
        )
    },
    {
        "issue_id": 6,
        "column": "gender",
        "problem_type": "inconsistent spelling / missing",
        "description/example": (
            "14 non-empty variants and 39 missing values "
            "in the raw dataset"
        ),
        "decision/rule": (
            "Mapped to Male, Female and Not specified; "
            "missing values became Not specified"
        ),
        "rows_affected": int(gender_changes),
        "assumption/comment": (
            "Gender was not inferred from employee names."
        )
    },
    {
        "issue_id": 7,
        "column": "city",
        "problem_type": "inconsistent spelling",
        "description/example": (
            "30 raw variants for 6 canonical locations"
        ),
        "decision/rule": (
            "Mapped all values to Kyiv, Lviv, Odesa, Dnipro, "
            "Kharkiv or Remote"
        ),
        "rows_affected": int(city_changes),
        "assumption/comment": (
            "Mappings were taken from reference_tables."
        )
    },
    {
        "issue_id": 8,
        "column": "employment_status",
        "problem_type": "inconsistent spelling",
        "description/example": (
            "14 raw variants for Active, Terminated and On leave"
        ),
        "decision/rule": (
            "Mapped values using reference_tables and created "
            "the is_active flag"
        ),
        "rows_affected": int(status_changes),
        "assumption/comment": (
            "Only status Active is included in the main salary KPIs."
        )
    },
    {
        "issue_id": 9,
        "column": "performance_score",
        "problem_type": "missing / outside valid range",
        "description/example": (
            "18 initial missing values, 25 zeros and 18 values of 6"
        ),
        "decision/rule": (
            "Converted values outside the allowed 1–5 range to NaN"
        ),
        "rows_affected": 61,
        "assumption/comment": (
            "Invalid scores were not capped or imputed because "
            "their true values are unknown."
        )
    }
])

display(cleaning_report)

,issue_id,column,problem_type,description/example,decision/rule,rows_affected,assumption/comment
0,1,employee_id,duplicates / invalid ID,1 exact duplicate; repeated E1010; invalid E10...,Removed the exact duplicate and restored 5 IDs...,6,Corrections were recorded in original_employee...
1,2,department_name,inconsistent spelling,"40 raw spellings, including MKT, Marketng, Tec...",Mapped all values to 8 canonical departments u...,169,Canonical department names from reference_tabl...
2,3,department_id vs department_name,mismatch,2 missing department_id values and 3 conflicts...,Recovered missing IDs from canonical names; fo...,5,department_id was treated as the primary struc...
3,4,salary,text format / zero / negative / missing / outlier,"Currency text formats, 2 missing, 2 zero, 2 ne...","Converted salary to numeric; excluded missing,...",8,"IQR outliers were flagged, not removed automat..."
4,5,hire_date,mixed date formats / invalid / future date,"Mixed ISO, dot and slash formats; 37 ambiguous...",Parsed dates by explicit rules; ambiguous slas...,40,Analysis date fixed at 2026-09-01. European fo...
5,6,gender,inconsistent spelling / missing,14 non-empty variants and 39 missing values in...,"Mapped to Male, Female and Not specified; miss...",164,Gender was not inferred from employee names.
6,7,city,inconsistent spelling,30 raw variants for 6 canonical locations,"Mapped all values to Kyiv, Lviv, Odesa, Dnipro...",168,Mappings were taken from reference_tables.
7,8,employment_status,inconsistent spelling,"14 raw variants for Active, Terminated and On ...",Mapped values using reference_tables and creat...,154,Only status Active is included in the main sal...
8,9,performance_score,missing / outside valid range,"18 initial missing values, 25 zeros and 18 val...",Converted values outside the allowed 1–5 range...,61,Invalid scores were not capped or imputed beca...


## 6. Export Cleaned Data and Summary Tables

Експортуємо очищену таблицю, cleaning report, результати фінальної перевірки, довідники та параметри аналізу в окремий Excel-файл.

In [ ]:
# Колонки для фінальної очищеної таблиці
export_columns = [
    "employee_id",
    "full_name",
    "department_id",
    "department_name",
    "job_title",
    "salary",
    "hire_date",
    "gender",
    "city",
    "employment_status",
    "performance_score",
    "tenure_months",
    "salary_level",
    "salary_vs_median",
    "is_active",
    "salary_outlier_flag",
    "salary_excluded_flag",
    "hire_date_issue_flag",
    "performance_score_issue_flag",
    "source_excel_row"
]

employees_cleaned_export = (
    employees_cleaned[export_columns]
    .copy()
)

# Таблиця з основними параметрами та припущеннями
analysis_parameters = pd.DataFrame({
    "parameter": [
        "analysis_date",
        "active_salary_q1",
        "active_salary_median",
        "active_salary_q3",
        "salary_iqr_upper_bound",
        "extreme_salary_threshold",
        "rows_in_raw_data",
        "rows_after_cleaning",
        "active_employees"
    ],
    "value": [
        analysis_date,
        active_salary_q1,
        active_salary_median,
        active_salary_q3,
        salary_upper_bound,
        2 * salary_upper_bound,
        len(employees_raw),
        len(employees_cleaned_export),
        employees_cleaned_export["is_active"].sum()
    ]
})

# Готуємо таблицю викидів із читабельною назвою зарплати
salary_outliers_export = salary_outliers.rename(
    columns={
        "salary_for_outlier_analysis": "salary"
    }
).copy()

# Усі підсумкові таблиці для експорту
summary_tables = {
    "summary_department": summary_by_department,
    "summary_city": summary_by_city,
    "summary_gender": summary_by_gender,
    "summary_job_title": summary_by_job_title,
    "salary_outliers": salary_outliers_export,
    "below_job_average": employees_below_job_average
}

output_file = "HR_Compensation_Cleaned.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
    datetime_format="yyyy-mm-dd"
) as writer:

    employees_cleaned_export.to_excel(
        writer,
        sheet_name="employees_cleaned",
        index=False
    )

    cleaning_report.to_excel(
        writer,
        sheet_name="cleaning_report",
        index=False
    )

    quality_check_table.to_excel(
        writer,
        sheet_name="quality_checks",
        index=False
    )

    reference_tables.to_excel(
        writer,
        sheet_name="reference_tables",
        index=False
    )

    analysis_parameters.to_excel(
        writer,
        sheet_name="analysis_parameters",
        index=False
    )

        # Експортуємо підсумкові таблиці
    for sheet_name, table in summary_tables.items():
        table.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )


# Перевіряємо створений файл
export_check = pd.ExcelFile(output_file)

print("ФАЙЛ УСПІШНО СТВОРЕНО")
print(f"Назва: {output_file}")
print(f"Аркуші: {export_check.sheet_names}")
print(
    "Розмір employees_cleaned:",
    employees_cleaned_export.shape
)

# Завантажуємо файл на Mac
files.download(output_file)

ФАЙЛ УСПІШНО СТВОРЕНО
Назва: HR_Compensation_Cleaned.xlsx
Аркуші: ['employees_cleaned', 'cleaning_report', 'quality_checks', 'reference_tables', 'analysis_parameters', 'summary_department', 'summary_city', 'summary_gender', 'summary_job_title', 'salary_outliers', 'below_job_average']
Розмір employees_cleaned: (199, 20)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side
)
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter


# Прибираємо порожні технічні колонки Unnamed із довідників
reference_tables_export = reference_tables.loc[
    :,
    ~reference_tables.columns.astype(str).str.startswith("Unnamed")
].copy()

formatted_output_file = (
    "HR_Compensation_Cleaned_Formatted.xlsx"
)

# Спочатку записуємо дані
with pd.ExcelWriter(
    formatted_output_file,
    engine="openpyxl",
    datetime_format="yyyy-mm-dd"
) as writer:

    employees_cleaned_export.to_excel(
        writer,
        sheet_name="employees_cleaned",
        index=False
    )

    cleaning_report.to_excel(
        writer,
        sheet_name="cleaning_report",
        index=False
    )

    quality_check_table.to_excel(
        writer,
        sheet_name="quality_checks",
        index=False
    )

    reference_tables_export.to_excel(
        writer,
        sheet_name="reference_tables",
        index=False
    )

    analysis_parameters.to_excel(
        writer,
        sheet_name="analysis_parameters",
        index=False
    )

        # Експортуємо підсумкові таблиці
    for sheet_name, table in summary_tables.items():
        table.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )


# Відкриваємо книгу для оформлення
workbook = load_workbook(formatted_output_file)

header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78"
)

header_font = Font(
    color="FFFFFF",
    bold=True
)

thin_gray_border = Border(
    bottom=Side(
        style="thin",
        color="D9E2F3"
    )
)

flag_fill = PatternFill(
    fill_type="solid",
    fgColor="FFF2CC"
)

problem_fill = PatternFill(
    fill_type="solid",
    fgColor="FCE4D6"
)


def format_header(worksheet):
    """Оформлення першого рядка."""

    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    worksheet.row_dimensions[1].height = 32
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions


def add_excel_table(
    worksheet,
    table_name,
    style_name="TableStyleMedium2"
):
    """Створює Excel Table на заповненому діапазоні."""

    table = Table(
        displayName=table_name,
        ref=worksheet.dimensions
    )

    style = TableStyleInfo(
        name=style_name,
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False
    )

    table.tableStyleInfo = style
    worksheet.add_table(table)

    # ---------- summary tables ----------
summary_table_names = {
    "summary_department": "SummaryDepartmentTable",
    "summary_city": "SummaryCityTable",
    "summary_gender": "SummaryGenderTable",
    "summary_job_title": "SummaryJobTitleTable",
    "salary_outliers": "SalaryOutliersTable",
    "below_job_average": "BelowJobAverageTable"
}

salary_columns = {
    "average_salary",
    "median_salary",
    "minimum_salary",
    "maximum_salary",
    "total_payroll",
    "salary",
    "average_salary_by_job",
    "difference_from_job_average"
}

for sheet_name, table_name in summary_table_names.items():
    ws = workbook[sheet_name]

    format_header(ws)
    add_excel_table(
        ws,
        table_name,
        "TableStyleMedium2"
    )

    ws.sheet_view.showGridLines = False
    ws.page_setup.orientation = "landscape"
    ws.freeze_panes = "A2"

    # Автоматично підбираємо ширину колонок
    for column_cells in ws.columns:
        column_letter = column_cells[0].column_letter

        maximum_length = max(
            len(str(cell.value))
            if cell.value is not None else 0
            for cell in column_cells
        )

        ws.column_dimensions[column_letter].width = min(
            maximum_length + 3,
            34
        )

    # Форматуємо числові показники
    for column_number, header_cell in enumerate(ws[1], start=1):
        header = header_cell.value

        if header in salary_columns:
            for row_number in range(2, ws.max_row + 1):
                ws.cell(
                    row=row_number,
                    column=column_number
                ).number_format = "#,##0.00"

        elif header == "difference_percent":
            for row_number in range(2, ws.max_row + 1):
                ws.cell(
                    row=row_number,
                    column=column_number
                ).number_format = "0.00"


# ---------- employees_cleaned ----------
ws = workbook["employees_cleaned"]

format_header(ws)
add_excel_table(ws, "EmployeesCleanedTable")

employee_widths = {
    "A": 14,   # employee_id
    "B": 24,   # full_name
    "C": 15,   # department_id
    "D": 22,   # department_name
    "E": 30,   # job_title
    "F": 14,   # salary
    "G": 14,   # hire_date
    "H": 16,   # gender
    "I": 15,   # city
    "J": 20,   # employment_status
    "K": 20,   # performance_score
    "L": 16,   # tenure_months
    "M": 16,   # salary_level
    "N": 20,   # salary_vs_median
    "O": 12,   # is_active
    "P": 19,   # salary_outlier_flag
    "Q": 20,   # salary_excluded_flag
    "R": 19,   # hire_date_issue_flag
    "S": 26,   # performance_score_issue_flag
    "T": 18    # source_excel_row
}

for column_letter, width in employee_widths.items():
    ws.column_dimensions[column_letter].width = width

for row in ws.iter_rows(min_row=2):
    row[5].number_format = "#,##0"
    row[6].number_format = "yyyy-mm-dd"

    for cell in row:
        cell.alignment = Alignment(
            vertical="center"
        )
        cell.border = thin_gray_border

    # Підсвічуємо прапорці проблем якості
    for column_index in [15, 16, 17, 18]:
        if row[column_index].value == 1:
            row[column_index].fill = flag_fill

    # Підсвічуємо пропуски в основних полях
    for column_index in [5, 6, 10]:
        if row[column_index].value is None:
            row[column_index].fill = problem_fill

ws.sheet_view.showGridLines = False
ws.page_setup.orientation = "landscape"
ws.page_setup.fitToWidth = 1
ws.page_setup.fitToHeight = 0
ws.sheet_properties.pageSetUpPr.fitToPage = True
ws.print_title_rows = "1:1"


# ---------- cleaning_report ----------
ws = workbook["cleaning_report"]

format_header(ws)
add_excel_table(
    ws,
    "CleaningReportTable",
    "TableStyleMedium4"
)

report_widths = {
    "A": 10,
    "B": 30,
    "C": 32,
    "D": 52,
    "E": 55,
    "F": 16,
    "G": 55
}

for column_letter, width in report_widths.items():
    ws.column_dimensions[column_letter].width = width

for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True
        )
    ws.row_dimensions[row[0].row].height = 72

ws.sheet_view.showGridLines = False
ws.page_setup.orientation = "landscape"
ws.page_setup.fitToWidth = 1
ws.page_setup.fitToHeight = 0
ws.sheet_properties.pageSetUpPr.fitToPage = True
ws.print_title_rows = "1:1"


# ---------- quality_checks ----------
ws = workbook["quality_checks"]

format_header(ws)
add_excel_table(
    ws,
    "QualityChecksTable",
    "TableStyleMedium9"
)

ws.column_dimensions["A"].width = 48
ws.column_dimensions["B"].width = 18
ws.sheet_view.showGridLines = False

for row in ws.iter_rows(min_row=2):
    row[0].alignment = Alignment(
        vertical="center"
    )
    row[1].alignment = Alignment(
        horizontal="center",
        vertical="center"
    )


# ---------- reference_tables ----------
ws = workbook["reference_tables"]

format_header(ws)
add_excel_table(
    ws,
    "ReferenceTablesTable",
    "TableStyleMedium2"
)

reference_widths = {
    "A": 16,
    "B": 26,
    "C": 62,
    "D": 22,
    "E": 62,
    "F": 24,
    "G": 58,
    "H": 24,
    "I": 68
}

for column_letter, width in reference_widths.items():
    ws.column_dimensions[column_letter].width = width

for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True
        )
    ws.row_dimensions[row[0].row].height = 60

ws.sheet_view.showGridLines = False
ws.page_setup.orientation = "landscape"
ws.page_setup.fitToWidth = 1
ws.page_setup.fitToHeight = 0
ws.sheet_properties.pageSetUpPr.fitToPage = True


# ---------- analysis_parameters ----------
ws = workbook["analysis_parameters"]

format_header(ws)
add_excel_table(
    ws,
    "AnalysisParametersTable",
    "TableStyleMedium9"
)

ws.column_dimensions["A"].width = 34
ws.column_dimensions["B"].width = 24
ws.sheet_view.showGridLines = False

# Формати для окремих параметрів
for row in range(2, ws.max_row + 1):
    parameter = ws.cell(row=row, column=1).value
    value_cell = ws.cell(row=row, column=2)

    if parameter == "analysis_date":
        value_cell.number_format = "yyyy-mm-dd"
    elif parameter in {
        "active_salary_q1",
        "active_salary_median",
        "active_salary_q3",
        "salary_iqr_upper_bound",
        "extreme_salary_threshold"
    }:
        value_cell.number_format = "#,##0"


# Робимо перший аркуш активним
workbook.active = workbook.sheetnames.index(
    "employees_cleaned"
)

workbook.save(formatted_output_file)

print("ОФОРМЛЕНИЙ ФАЙЛ СТВОРЕНО")
print(formatted_output_file)
print(workbook.sheetnames)

files.download(formatted_output_file)

ОФОРМЛЕНИЙ ФАЙЛ СТВОРЕНО
HR_Compensation_Cleaned_Formatted.xlsx
['employees_cleaned', 'cleaning_report', 'quality_checks', 'reference_tables', 'analysis_parameters', 'summary_department', 'summary_city', 'summary_gender', 'summary_job_title', 'salary_outliers', 'below_job_average']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
csv_output_file = "employees_cleaned.csv"

employees_cleaned_export.to_csv(
    csv_output_file,
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8"
)

print(
    f"CSV створено: {csv_output_file}; "
    f"рядків: {len(employees_cleaned_export)}"
)

files.download(csv_output_file)

CSV створено: employees_cleaned.csv; рядків: 199


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task 5 — A/B Test Planning

## 1. Hypothesis and Primary Metric

HR-команда планує порівняти два формати адаптації нових співробітників:

- **Control group:** чинний процес адаптації;
- **Test group:** новий формат із ментором і коротшим списком завдань.

Працівники мають бути випадково розподілені між групами у співвідношенні 50/50.

### Hypotheses

- **H₀:** частка працівників, які успішно проходять випробувальний термін, однакова в контрольній і тестовій групах:  
  \(p_{test} = p_{control}\).

- **H₁:** частка працівників, які успішно проходять випробувальний термін, відрізняється між групами:  
  \(p_{test} \neq p_{control}\).

Очікуваний напрям ефекту — збільшення показника з 77% до 85%, тобто на 8 відсоткових пунктів.

### Primary Metric

Основна метрика — **probation pass rate**, частка нових співробітників, які успішно пройшли випробувальний термін:

\[
\text{Probation Pass Rate} =
\frac{\text{Кількість працівників, які пройшли випробувальний термін}}
{\text{Кількість працівників, які завершили період спостереження}}
\]

До знаменника потрібно включати лише працівників, для яких випробувальний термін уже завершився і результат відомий.

## 2. Sample Size Calculation

Для розрахунку вибірки використовуємо:

- базовий рівень метрики: 77%;
- очікуваний рівень у тестовій групі: 85%;
- мінімальний ефект для виявлення: 8 відсоткових пунктів;
- рівень значущості: \(\alpha = 0.05\);
- статистичну потужність: \(1-\beta = 0.80\);
- двосторонню перевірку гіпотези;
- рівний розподіл працівників між групами 50/50.

In [ ]:
import math

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize


# Вхідні параметри тесту
baseline_rate = 0.77
expected_rate = 0.85
alpha = 0.05
power = 0.80
group_ratio = 1

# Розмір ефекту для порівняння двох пропорцій
effect_size = abs(
    proportion_effectsize(
        expected_rate,
        baseline_rate
    )
)

# Розрахунок необхідної вибірки для однієї групи
sample_size_raw = NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=group_ratio,
    alternative="two-sided"
)

sample_size_per_group = math.ceil(sample_size_raw)
total_sample_size = sample_size_per_group * 2

print("РОЗРАХУНОК РОЗМІРУ ВИБІРКИ")
print(f"Базовий рівень: {baseline_rate:.0%}")
print(f"Очікуваний рівень: {expected_rate:.0%}")
print(
    "Мінімальний ефект:",
    f"{(expected_rate - baseline_rate) * 100:.0f} в.п."
)
print(f"Рівень значущості: {alpha}")
print(f"Статистична потужність: {power:.0%}")
print(f"Розрахований розмір групи: {sample_size_raw:.2f}")
print(f"Необхідно на одну групу: {sample_size_per_group}")
print(f"Загальна вибірка: {total_sample_size}")

РОЗРАХУНОК РОЗМІРУ ВИБІРКИ
Базовий рівень: 77%
Очікуваний рівень: 85%
Мінімальний ефект: 8 в.п.
Рівень значущості: 0.05
Статистична потужність: 80%
Розрахований розмір групи: 373.68
Необхідно на одну групу: 374
Загальна вибірка: 748


## 3. Estimated Test Duration

Компанія наймає в середньому 25 нових співробітників на місяць. За рівного розподілу 50/50 приблизно 12–13 працівників щомісяця потраплятимуть до кожної групи.

Орієнтовна тривалість набору вибірки розраховується як відношення загального необхідного розміру вибірки до середньої кількості нових наймів на місяць.

In [ ]:
average_hires_per_month = 25

recruitment_months_raw = (
    total_sample_size
    / average_hires_per_month
)

recruitment_months = math.ceil(
    recruitment_months_raw
)

hires_per_group_per_month = (
    average_hires_per_month / 2
)

print("ОРІЄНТОВНА ТРИВАЛІСТЬ ТЕСТУ")
print(
    "Середня кількість нових наймів на місяць:",
    average_hires_per_month
)
print(
    "Приблизна кількість учасників на групу за місяць:",
    hires_per_group_per_month
)
print(
    "Розрахована тривалість набору:",
    f"{recruitment_months_raw:.2f} місяця"
)
print(
    "Орієнтовна тривалість набору після округлення:",
    f"{recruitment_months} місяців"
)

ОРІЄНТОВНА ТРИВАЛІСТЬ ТЕСТУ
Середня кількість нових наймів на місяць: 25
Приблизна кількість учасників на групу за місяць: 12.5
Розрахована тривалість набору: 29.92 місяця
Орієнтовна тривалість набору після округлення: 30 місяців


## 4. Risks and Limitations

Під час планування та проведення A/B-тесту потрібно врахувати такі ризики й обмеження:

1. **Тривалий набір вибірки.**  
   Для отримання 748 учасників за швидкості 25 наймів на місяць потрібно приблизно 30 місяців. За цей час можуть змінитися умови найму, структура компанії, керівники, компенсації та сам процес адаптації.

2. **Період очікування результату.**  
   Результат основної метрики стає відомим лише після завершення випробувального терміну. Тому до 30 місяців набору необхідно додати тривалість випробувального терміну для останньої когорти.

3. **Сезонність найму.**  
   Кількість і характеристики нових працівників можуть відрізнятися залежно від місяця або кварталу. Тест повинен охоплювати однакові календарні періоди для обох груп.

4. **Різна структура посад і відділів.**  
   Імовірність проходження випробувального терміну може залежати від посади, відділу, рівня seniority або міста. Випадковий розподіл потрібно за можливості стратифікувати за ключовими категоріями, насамперед за посадою або відділом.

5. **Нерівномірний розподіл між групами.**  
   Працівники повинні випадково розподілятися між контрольною і тестовою групами у співвідношенні 50/50. Потрібно контролювати фактичну чисельність і склад обох груп.

6. **Вплив конкретного ментора.**  
   Результат може залежати не лише від нового формату, а й від досвіду та залученості конкретного ментора. Необхідно стандартизувати роботу менторів або враховувати ментора під час аналізу.

7. **Перетікання між варіантами тесту.**  
   Працівники або керівники з контрольної групи можуть частково використовувати елементи нового процесу. Це зменшить різницю між групами та ускладнить оцінювання ефекту.

8. **Неповні результати та вибуття.**  
   Частина працівників може звільнитися до завершення випробувального терміну або мати невідомий результат. Правила обробки таких випадків потрібно визначити до початку тесту. За ризику втрати 10% спостережень вибірку варто збільшити приблизно до 416 учасників на групу.

9. **Єдине визначення метрики.**  
   Потрібно заздалегідь зафіксувати, що саме означає «успішно пройшов випробувальний термін», дату початку й завершення спостереження та правила врахування переведень, відпусток і дострокових звільнень.

10. **Передчасна перевірка результатів.**  
    Не слід багаторазово перевіряти статистичну значущість до накопичення запланованої вибірки, оскільки це підвищує ризик хибнопозитивного висновку.

### Planning Conclusion

За поточної швидкості найму класичний A/B-тест потребуватиме приблизно 30 місяців лише для набору вибірки, тому його проведення буде тривалим. Перед запуском HR-команді варто оцінити можливість залучити більше підрозділів або локацій, збільшити потік учасників чи переглянути мінімальний ефект, який необхідно виявити. Змінювати рівень значущості, потужність або напрям тесту слід лише до його початку та з чітким обґрунтуванням.